# Urbanride — Bronze CSV Ingestion Pipeline
## Notebook: urbanride_01
## Method: COPY INTO (incremental, file-tracked)

**What this notebook does:**
- Ingests CSV files from the UC Volume into Bronze Delta tables
- Targets: `customers` · `drivers` · `payments`
- Uses COPY INTO — only picks up new files on every run
- Safe to run daily — no duplicates, no full reload

**Datasets:**
| Dataset | Format | Source Path | Target Table |
|---|---|---|---|
| Customers | CSV | `/Volumes/urbanride/bronze/raw_files/customers/` | `urbanride.bronze.customers` |
| Drivers | CSV | `/Volumes/urbanride/bronze/raw_files/drivers/` | `urbanride.bronze.drivers` |
| Payments | CSV | `/Volumes/urbanride/bronze/raw_files/payments/` | `urbanride.bronze.payments` |

**Run schedule:** Daily — at 01:00 AM

## Urbanride_01 — Bronze CSV Ingestion
**Method:** COPY INTO  
**Inputs:** `/Volumes/Urbanride/bronze/raw_files/` — customers, drivers, payments  
**Outputs:** `Urbanride.bronze.customers` · `Urbanride.bronze.drivers` · `Urbanride.bronze.payments`  
**Schedule:** runs daily at 01:00 AM
             picks up whatever files landed in the Volume
             doesn't care how they got there   
**Owner:** Vishal Bankar (Data Engineering) 

In [0]:
# ---- CATALOG CONFIG ----
CATALOG   = 'urbanride'
BRONZE    = f'{CATALOG}.bronze'
BASE_PATH = f'/Volumes/{CATALOG}/bronze/raw_files'

# ---- SOURCE PATHS ----
CUSTOMERS_PATH = f'{BASE_PATH}/customers/'
DRIVERS_PATH   = f'{BASE_PATH}/drivers/'
PAYMENTS_PATH  = f'{BASE_PATH}/payments/'

# ---- CHECKPOINT — COPY INTO tracks files internally in Delta log ----
# No separate checkpoint needed — Delta handles this automatically

spark.sql(f'USE CATALOG {CATALOG}')

print(f'Catalog  : {CATALOG}')
print(f'Bronze   : {BRONZE}')
print(f'Source   : {BASE_PATH}')
print('Config ready.')


Catalog  : urbanride
Bronze   : urbanride.bronze
Source   : /Volumes/urbanride/bronze/raw_files
Config ready.


In [0]:
# Create empty Delta tables first if they don't exist
# COPY INTO requires the target table to exist before loading
# IF NOT EXISTS — safe to re-run, won't overwrite existing data
print('Creating Bronze Delta tables if not exist...')

# -----------------  CUSTOMERS --------------------------------------------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.customers (
        customer_id              STRING,
        name                     STRING,
        city                     STRING,
        signup_date              DATE,
        device_type              STRING,
        referral_source          STRING,
        referred_by_customer_id  STRING,
        rating                   DOUBLE,
        is_churned               BOOLEAN,
        churn_date               DATE,
        _ingested_at             TIMESTAMP
    )
    USING DELTA
    COMMENT 'Bronze raw customer records — append only'
""")
print('urbanride.bronze.customers — ready')

# ----------------- DRIVERS  --------------------------------------------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.drivers (
        driver_id        STRING,
        name             STRING,
        city             STRING,
        vehicle_type     STRING,
        rating           DOUBLE,
        trips_completed  INT,
        joining_date     DATE,
        status           STRING,
        _ingested_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Bronze raw driver master records — append only'
""")
print('urbanride.bronze.drivers — ready')

# ─----------------- PAYMENTS --------------------------------------------
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE}.payments (
        payment_id        STRING,
        trip_id           STRING,
        customer_id       STRING,
        amount            DOUBLE,
        payment_method    STRING,
        payment_timestamp TIMESTAMP,
        payment_status    STRING,
        card_hash         STRING,
        is_fraud_card     BOOLEAN,
        _ingested_at      TIMESTAMP
    )
    USING DELTA
    COMMENT 'Bronze raw payment records — append only'
""")
print('urbanride.bronze.payments — ready')
print()
print('All Bronze CSV tables ready for COPY INTO.')


Creating Bronze Delta tables if not exist...
zipcab.bronze.customers — ready
zipcab.bronze.drivers — ready
zipcab.bronze.payments — ready

All Bronze CSV tables ready for COPY INTO.


In [0]:
# COPY INTO — incremental file ingestion
# Delta tracks which files have been loaded in the transaction log
# Re-running this cell will only pick up NEW files — never reloads old ones
print('Loading customers...')
import time
t0 = time.time()

spark.sql(f"""
    COPY INTO {BRONZE}.customers
    FROM (
        SELECT
            customer_id,
            name,
            city,
            CAST(signup_date AS DATE)   AS signup_date,
            device_type,
            referral_source,
            referred_by_customer_id,
            CAST(rating AS DOUBLE) AS rating,
            CAST(is_churned AS BOOLEAN) AS is_churned,
            CAST(churn_date AS DATE)  AS churn_date,
            current_timestamp()  AS _ingested_at
        FROM '{CUSTOMERS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header'      = 'true',
        'inferSchema' = 'false',
        'nullValue'   = ''
    )
    COPY_OPTIONS (
        'mergeSchema' = 'true',
        'force'       = 'false'
    )
""")

n = spark.table(f'{BRONZE}.customers').count() # count rows been processed
print(f'Rows in bronze.customers : {n:,}')
print(f'Time : {round(time.time()-t0,2)}s')
print('Customers loaded.')


Loading customers...
Rows in bronze.customers : 89,963
Time : 20.22s
Customers loaded.


In [0]:
# Drivers is a master file — loaded once
# COPY INTO will skip it on subsequent runs since file hash is already tracked
print('Loading drivers...')
t0 = time.time()

spark.sql(f"""
    COPY INTO {BRONZE}.drivers
    FROM (
        SELECT
            driver_id,
            name,
            city,
            vehicle_type,
            CAST(rating AS DOUBLE)          AS rating,
            CAST(trips_completed AS INT)    AS trips_completed,
            CAST(joining_date AS DATE)      AS joining_date,
            status,
            current_timestamp()             AS _ingested_at
        FROM '{DRIVERS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header'      = 'true',
        'inferSchema' = 'false',
        'nullValue'   = ''
    )
    COPY_OPTIONS (
        'mergeSchema' = 'true',
        'force'       = 'false'
    )
""")

n = spark.table(f'{BRONZE}.drivers').count()
print(f'  Rows in bronze.drivers : {n:,}')
print(f'  Time                   : {round(time.time()-t0,2)}s')
print('  Drivers loaded.')


Loading drivers...
  Rows in bronze.drivers : 20,000
  Time                   : 4.29s
  Drivers loaded.


In [0]:
print('Loading payments...')
t0 = time.time()

spark.sql(f"""
    COPY INTO {BRONZE}.payments
    FROM (
        SELECT
            payment_id,
            trip_id,
            customer_id,
            CAST(amount AS DOUBLE)              AS amount,
            payment_method,
            CAST(payment_timestamp AS TIMESTAMP) AS payment_timestamp,
            payment_status,
            card_hash,
            CAST(is_fraud_card AS BOOLEAN)      AS is_fraud_card,
            current_timestamp()                 AS _ingested_at
        FROM '{PAYMENTS_PATH}'
    )
    FILEFORMAT = CSV
    FORMAT_OPTIONS (
        'header'      = 'true',
        'inferSchema' = 'false',
        'nullValue'   = ''
    )
    COPY_OPTIONS (
        'mergeSchema' = 'true',
        'force'       = 'false'
    )
""")

n = spark.table(f'{BRONZE}.payments').count()
print(f'  Rows in bronze.payments : {n:,}')
print(f'  Time                    : {round(time.time()-t0,2)}s')
print('  Payments loaded.')


Loading payments...
  Rows in bronze.payments : 19,600,753
  Time                    : 24.13s
  Payments loaded.


In [0]:
# Verify row counts, schema, and sample data for all 3 tables
print('=== BRONZE CSV INGESTION VERIFICATION ===')
print()

tables = [
    (f'{BRONZE}.customers', 'Customers'),
    (f'{BRONZE}.drivers',   'Drivers'),
    (f'{BRONZE}.payments',  'Payments'),
]

print(f'  {"Table":<35} {"Rows":>15}  {"Columns":>10}')
print('  ' + '-'*65)
for tbl, name in tables:
    df  = spark.table(tbl)
    n   = df.count()
    cols = len(df.columns)
    print(f'  {tbl:<35} {n:>15,}  {cols:>10}')

print()

# Check COPY INTO file tracking — how many files loaded
print('File tracking (COPY INTO history):')
for tbl, name in tables:
    history = spark.sql(f'DESCRIBE HISTORY {tbl}').filter('operation = "COPY INTO"')
    n_ops = history.count()
    print(f'  {name:<15}: {n_ops} COPY INTO operation(s) recorded')

print()
print('Sample from bronze.customers:')
spark.table(f'{BRONZE}.customers').limit(3).show(truncate=False)

print('urbanride_01 complete — Bronze CSV tables loaded and verified.')


=== BRONZE CSV INGESTION VERIFICATION ===

  Table                                          Rows     Columns
  -----------------------------------------------------------------
  urbanride.bronze.customers                   89,963          11
  urbanride.bronze.drivers                     20,000           9
  urbanride.bronze.payments                19,600,753          10

File tracking (COPY INTO history):
  Customers      : 1 COPY INTO operation(s) recorded
  Drivers        : 1 COPY INTO operation(s) recorded
  Payments       : 1 COPY INTO operation(s) recorded

Sample from bronze.customers:
+------------------------------------+----------------+---------+-----------+-----------+---------------+-----------------------+------+----------+----------+--------------------------+
|customer_id                         |name            |city     |signup_date|device_type|referral_source|referred_by_customer_id|rating|is_churned|churn_date|_ingested_at              |
+--------------------------